# Course 14 lab — OpenSpec enterprise change governance

Northstar has an approved document-processing baseline and two concurrent changes. We will create a candidate target, inspect conflicts, validate bounded work, inject failures, and prove that archive remains externally governed. The notebook is deterministic, credential-free, and uses only observable decisions—never private model reasoning.

## Success criteria and safety boundary

A successful run preserves the baseline, records both semantic conflicts, maps every task to a bounded AWU, covers changed and regression claims independently, and reports only `READY_FOR_OWNER_REVIEW`. The fixture is synthetic; it cannot approve, archive, or release anything.

In [ ]:
import importlib.util
import sys
from pathlib import Path

module_spec = importlib.util.spec_from_file_location('course14_notebook_lab', Path('lab.py'))
lab = importlib.util.module_from_spec(module_spec)
sys.modules[module_spec.name] = lab
module_spec.loader.exec_module(lab)

baseline = lab.load_baseline()
context = lab.load_context()
snapshot = lab.load_snapshot()
reference = lab.load_reference()
print(snapshot['release'], snapshot['resolved_commit'][:12], baseline['revision'], context['id'])

## Baseline + delta produces a candidate—not approved truth

`apply_change` is pure: it returns a candidate target and never writes the baseline. This lets us inspect semantic impact before any integration decision.

In [ ]:
change = lab.primary_change(reference)
target = lab.apply_change(baseline, change)
before = lab.requirement_index(baseline)['REQ-DOC-014']
after = lab.requirement_index(target)['REQ-DOC-014']
print('before:', before['revision'], before['text'])
print('candidate:', after['revision'], after['text'])
print('baseline still:', baseline['revision'])
assert baseline['revision'] == 'document-processing@4-training'
assert target['revision'] == 'candidate-target-not-integrated'

## Concurrent change graph

Both changes modify `REQ-DOC-014@3`. The reference does not pretend they are compatible: it records a shared-requirement conflict and an authority conflict, then blocks AI-2331 pending rebase.

In [ ]:
for conflict in reference['conflicts']:
    print(conflict['type'], '→', conflict['disposition'], 'blocked:', conflict['blocked_change'])
assert lab.change_by_id(reference, 'AI-2331')['state'] == 'BLOCKED_PENDING_REBASE'

## Validate the governed reference

The validator checks pinned framework semantics, exact context and baseline digests, full prior requirement state, authority, concurrency, AWUs, evidence, multi-repository inheritance, and integration state.

In [ ]:
reference_findings = lab.review_bundle(baseline, context, snapshot, reference)
print(lab.selection_state(reference_findings), len(reference_findings))
assert reference_findings == []
assert reference['integration']['state'] == 'OWNER_REVIEW_REQUIRED'

## Inspect claim-specific evidence

Each AWU carries evidence for its own claims. The union includes both delta evidence and regression evidence, while the implementation agent remains unable to attest its own results.

In [ ]:
for unit in reference['work_units']:
    print(unit['id'], '→', unit['required_evidence'])
evidence_sets = {tuple(unit['required_evidence']) for unit in reference['work_units']}
assert len(evidence_sets) == len(reference['work_units'])

## Failure injection: the unsafe candidate

The candidate combines stale context, hidden concurrency, authority laundering, weak evidence, unsafe archive, child override, and agent self-approval. It must block for independent reasons.

In [ ]:
candidate = lab.load_candidate()
candidate_snapshot = candidate.pop('_candidate_snapshot')
candidate_findings = lab.review_bundle(baseline, context, candidate_snapshot, candidate)
print(lab.selection_state(candidate_findings), len({item.code for item in candidate_findings}))
print(sorted({item.code for item in candidate_findings}))
assert lab.selection_state(candidate_findings) == 'BLOCKED'

## Labelled evaluation

Golden scenarios measure decision routing. Mutation cases measure whether one injected control failure produces its expected code. Neither is a general benchmark.

In [ ]:
golden = lab.run_conformance_suite()
mutations = lab.run_evaluation()
print('golden:', f"{golden['passed']}/{golden['total']}")
print('mutations:', f"{mutations['passed']}/{mutations['total']}")
assert golden['passed'] == golden['total'] == 10
assert mutations['passed'] == mutations['total'] == 53

## Compare the simpler baseline

A ticket-plus-PR baseline has lower process cost but cannot reliably show semantic concurrency, affected invariants, or approved baseline evolution. Use the full change package when consequence, uncertainty, shared contracts, migration, or parallel work justifies it—not for every typo.

## Production upgrade

Replace fixture identities with authenticated principals; JSON ownership with policy services; local hashes with signed provenance; synthetic records with independent CI/evaluation/security/runtime attestations; and boolean integration state with a transactional, exact-revision, single-use authorization receipt. Keep production release separate from spec archive.

## Exercises

1. Add a removal with migration and zero-reader evidence.
2. Split automatic persistence into a separately blocked change.
3. Add an NFR interaction across different requirement IDs.
4. Design a single-use integration receipt and its replay test.
5. Explain when OpenSpec change governance plus a Spec Kit implementation flow adds value—and when it is process theatre.